In [1]:
from pathlib import Path 

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox


In [2]:
from adjustText import adjust_text

In [3]:
import ipywidgets as widgets

In [4]:
print(Path.cwd())

d:\Wu\2026\Project Portfolio\002 Project\grocery_stores_access\notebooks


In [5]:
PROCESSED_DIR = Path("../data/processed")

In [6]:
county_boundary = gpd.read_file(PROCESSED_DIR / "county_boundary.gpkg")
stores = gpd.read_file(PROCESSED_DIR / "grocery_store_nodes.gpkg")
G = ox.load_graphml(PROCESSED_DIR / "kootenai_drive.graphml")

In [7]:
nodes, edges = ox.graph_to_gdfs(G)
roads = edges

In [8]:
cities = gpd.read_file(PROCESSED_DIR / "kootenai_cities.gpkg")

In [9]:
cities

,STATEFP,PLACEFP,PLACENS,GEOIDFQ,GEOID,city,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,16,17830,02585569,1600000US1617830,1617830,Conkling Park,Conkling Park CDP,ID,Idaho,57,2705973,0,"MULTIPOLYGON (((517018.408 5250112.31, 517497...."
1,16,88480,02412309,1600000US1688480,1688480,Worley,Worley city,ID,Idaho,25,502488,0,"MULTIPOLYGON (((505881.753 5250288.637, 506273..."
2,16,35200,02410708,1600000US1635200,1635200,Harrison,Harrison city,ID,Idaho,25,9231198,2176062,"MULTIPOLYGON (((517587.951 5257215.899, 517660..."
3,16,68950,02585592,1600000US1668950,1668950,Rockford Bay,Rockford Bay CDP,ID,Idaho,57,9778278,0,"MULTIPOLYGON (((506518.161 5261806.717, 506428..."
4,16,27550,02410496,1600000US1627550,1627550,Fernan Lake Village,Fernan Lake Village city,ID,Idaho,25,197627,2689,"MULTIPOLYGON (((518702.482 5279839.263, 518860..."
5,16,16750,02410187,1600000US1616750,1616750,Coeur d'Alene,Coeur d'Alene city,ID,Idaho,25,43691994,1976736,"MULTIPOLYGON (((511182.052 5283066.24, 511182...."
6,16,39070,02410802,1600000US1639070,1639070,Huetter,Huetter city,ID,Idaho,25,117471,0,"MULTIPOLYGON (((511131.877 5283526.82, 511179...."
7,16,77050,02411974,1600000US1677050,1677050,State Line,State Line city,ID,Idaho,25,285215,0,"MULTIPOLYGON (((497046.14 5283452.781, 497068...."
8,16,64810,02411475,1600000US1664810,1664810,Post Falls,Post Falls city,ID,Idaho,25,49621216,70897,"MULTIPOLYGON (((508530.246 5285326.007, 508530..."
9,16,20350,02410289,1600000US1620350,1620350,Dalton Gardens,Dalton Gardens city,ID,Idaho,25,6168849,13365,"MULTIPOLYGON (((515988.725 5287488.891, 515988..."


In [10]:
all_service_areas = gpd.read_file(
    PROCESSED_DIR / "grocery_service_areas.gpkg"
)

In [49]:
all_service_areas = all_service_areas.rename(columns={"name": "store"})

In [50]:
all_service_areas.head()

,store,city,minutes,geometry
0,Grocery Outlet,Coeur d'Alene,10,"POLYGON ((-116.95645 47.68717, -116.95749 47.6..."
1,Grocery Outlet,Coeur d'Alene,20,"POLYGON ((-117.02485 47.65923, -117.02505 47.6..."
2,Grocery Outlet,Coeur d'Alene,30,"POLYGON ((-117.04992 47.43279, -117.05017 47.4..."
3,Little Town Market,Athol,10,"POLYGON ((-116.83854 47.94133, -116.83857 47.9..."
4,Little Town Market,Athol,20,"POLYGON ((-116.9091 47.77633, -116.91005 47.77..."


In [12]:
all_service_areas.shape

(90, 4)

In [13]:
all_service_areas["minutes"].value_counts()

minutes
10    30
20    30
30    30
Name: count, dtype: int64

In [14]:
all_service_areas = all_service_areas.to_crs(roads.crs)
cities = cities.to_crs(roads.crs)
stores = stores.to_crs(roads.crs)
county_boundary = county_boundary.to_crs(roads.crs)

In [15]:
print(roads.crs)
print(all_service_areas.crs)
print(cities.crs)
print(stores.crs)
print(county_boundary.crs)

epsg:4326
epsg:4326
epsg:4326
EPSG:4326
epsg:4326


In [16]:
service10 = all_service_areas[
    all_service_areas["minutes"] == 10
]

In [17]:
service20 = all_service_areas[
    all_service_areas["minutes"] == 20
]

In [18]:
service30 = all_service_areas[
    all_service_areas["minutes"] == 30
]

## Filter Map Display
### One filter based on drive time: 10min, 20min, and 30min
### The other filter based on store name

In [51]:
all_service_areas.columns

Index(['store', 'city', 'minutes', 'geometry'], dtype='str')

In [53]:
# Create Dropdown options
stores_list = (
    ["All Stores"]
    + sorted(all_service_areas["store"].unique())
)

minutes_list = [
    "All",
    10,
    20,
    30
]

In [54]:
store_dropdown = widgets.Dropdown(
    options=stores_list,
    value="All Stores",
    description="Store:"
)

In [55]:
minute_dropdown = widgets.Dropdown(
    options=minutes_list,
    value="All",
    description="Minutes:"
)

In [56]:
display(
    store_dropdown,
    minute_dropdown
)

Dropdown(description='Store:', options=('All Stores', 'Albertsons', 'C & C Grocery', 'Flour Mill Natural Foods…

Dropdown(description='Minutes:', options=('All', 10, 20, 30), value='All')

In [70]:
stores = stores.rename(columns={"name": "store"})

In [71]:
def draw_map(selected_minutes, selected_store):  
    filtered = all_service_areas.copy()
    if selected_store != "All Stores":
        filtered = filtered[
            filtered["store"] == selected_store
        ]

    if selected_minutes != "All":
        filtered = filtered[
            filtered["minutes"] == selected_minutes
        ]
    filtered_stores = stores.copy()
    if selected_store != "All Stores":
        filtered_stores = filtered_stores[
            filtered_stores["store"] == selected_store
        ]

    fig, ax = plt.subplots(figsize=(12, 12))

    county_boundary.plot(ax=ax, color="lightgray", linewidth=1, zorder=1)
    roads.plot(ax=ax, color="darkgray", linewidth=0.5, zorder=2)

    filtered.plot(ax=ax, alpha=0.4, zorder=3)
    
    filtered_stores.plot(
        ax=ax,
        marker="*",
        color="red",
        markersize=20,
        zorder=4
    )

    texts = []
    for _, row in cities.iterrows():
        label_point = row.geometry.representative_point()

        texts.append(
            ax.text(
                label_point.x,
                label_point.y,
                row["city"],
                fontsize=9
            )
        )
    adjust_text(texts)

    ax.set_title(f"Service Areas of {selected_store}-{selected_minutes}min")
    plt.show()

In [72]:
ui = widgets.interactive(
    draw_map,
    selected_store=store_dropdown,
    selected_minutes=minute_dropdown
)

display(ui)

interactive(children=(Dropdown(description='Minutes:', index=1, options=('All', 10, 20, 30), value=10), Dropdo…